In [ ]:
#Henry Sampers: 23363819 Fuwhad Akinwale: 23362227 David Bradshaw: 23377569
#Does the code run: TBD
#Link to third part source: 


#Imports
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

#setting dir paths
data_dir = Path("data/state-farm")
csv_path = data_dir / "driver_imgs_list.csv"
train_dir = data_dir / "imgs" / "train"

#Checks to ensure data is there
print("CSV exists:", csv_path.exists())
print("Training images exist:", train_dir.exists())

#printing classes and counts
df = pd.read_csv(csv_path)
display(df.head())
print(df["classname"].value_counts().sort_index())

In [ ]:
#select 6 random examples from the dataset they will always be same due to the random_state parameter
examples = df.sample(6, random_state=42)

#pairs each image with its data in the driver_imgs_list.csv file and displays it in a 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (_, row) in zip(axes.flat, examples.iterrows()):
    image_path = train_dir / row["classname"] / row["img"]
    with Image.open(image_path) as image:
        ax.imshow(image)

    ax.set_title(f'{row["classname"]} | driver {row["subject"]}')
    ax.axis("off")
#actually displays the images
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

#making the 4 phone classes into one positive label for the binary classification task
phone_classes = {"c1", "c2", "c3", "c4"}
df["phone_use"] = df["classname"].isin(phone_classes).astype(int)
print("Binary label counts:")
print(df["phone_use"].value_counts().sort_index())

#reserve five complete drivers for the final test.
#these images must not influence model selection or hyperparameter tuning
test_split = GroupShuffleSplit(n_splits=1, test_size=5, random_state=42)
development_idx, test_idx = next(
    test_split.split(df, groups=df["subject"])
)

development_df = df.iloc[development_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
test_drivers = set(test_df["subject"])

print(
    f"Final test: {len(test_df)} images, "
    f"{len(test_drivers)} drivers, "
    f"{test_df['phone_use'].mean():.1%} phone use"
)

#validate on five different groups of drivers from the development data
#each driver is in the validation set once and never shares a folds training set
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
cv_folds = []
seen_validation_drivers = set()

for fold_number, (train_idx, val_idx) in enumerate(
    cv.split(
        development_df,
        y=development_df["phone_use"],
        groups=development_df["subject"],
    ),
    start=1,
):
    fold_train_df = development_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = development_df.iloc[val_idx].reset_index(drop=True)

    train_drivers = set(fold_train_df["subject"])
    val_drivers = set(fold_val_df["subject"])

    #detect accidental driver overlap before any model is trained
    #driver overlap could lead to overfitting of the model and overestimation of its performance
    assert train_drivers.isdisjoint(val_drivers)
    assert train_drivers.isdisjoint(test_drivers)
    assert val_drivers.isdisjoint(test_drivers)
    assert val_drivers.isdisjoint(seen_validation_drivers)

    #each fold needs both labels for binary performance metrics
    assert fold_train_df["phone_use"].nunique() == 2
    assert fold_val_df["phone_use"].nunique() == 2

    seen_validation_drivers.update(val_drivers)
    cv_folds.append((fold_train_df, fold_val_df))

    print(
        f"Fold {fold_number}: "
        f"{len(fold_train_df)} train images, "
        f"{len(fold_val_df)} validation images, "
        f"{fold_val_df['phone_use'].mean():.1%} validation phone use"
    )

assert seen_validation_drivers == set(development_df["subject"])
print("All development drivers are validated once; test drivers remain separate.")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class DriverPhoneDataset(Dataset):
    #Match each CSV row to an image and its binary phone use label

    def __init__(self, frame, image_root, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        #Kaggle stores each image inside the folder for its original class
        image_path = self.image_root / row["classname"] / row["img"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")  #Ensure every image has three channels
            image = self.transform(image)

        #1 means phone use 
        #0 means another activity
        label = int(row["phone_use"])
        return image, label

#Match the papers image size and normalization for the baseline experiment
preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

#just using the first fold for now, later we will loop over all 5 folds
#this is just for image display and data loader sanity checks
preview_train_df, preview_val_df = cv_folds[0]

train_dataset = DriverPhoneDataset(preview_train_df, train_dir, preprocess)
val_dataset = DriverPhoneDataset(preview_val_df, train_dir, preprocess)

#Shuffle training batches keep validation order fixed for repeatable evaluation
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=0
)

#Check one batch before connecting the loader to a model
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels in batch:", labels.tolist())

In [ ]:
import torch

#Select examples of both binary labels from the training set
phone_indices = preview_train_df.index[preview_train_df["phone_use"] == 1][:3].tolist()
other_indices = preview_train_df.index[preview_train_df["phone_use"] == 0][:3].tolist()
selected_indices = phone_indices + other_indices

#The tensors were normalized for training; reverse that only for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, index in zip(axes.flat, selected_indices):
    image, label = train_dataset[index]
    image = (image * std + mean).clamp(0, 1)

    #Matplotlib expects height, width, channels. PyTorch uses channels first
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title("Phone use" if label == 1 else "Other activity")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Imports for building and training the model
import time   # time each epoch
import copy   # keep a copy of the best model weights
import torch
import torch.nn as nn
import torch.nn.functional as F

# Use the GPU if one is available, otherwise fall back to the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Fix the random seeds so the starting weights and shuffle order are the same each run
torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.manual_seed_all(42)

In [ ]:
class Bottleneck(nn.Module):
    # Output channels are 4x the block's inner width (e.g. 64 -> 256)
    expansion = 4

    def __init__(self, in_channels, width, stride=1, downsample=None):
        super().__init__()

        # 1x1 conv: shrink the channels to the inner width to cut computation
        self.conv1 = nn.Conv2d(in_channels, width, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)

        # 3x3 conv: the main spatial feature extraction; stride 2 here halves the image size
        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width)

        # 1x1 conv: expand the channels back up (width * 4)
        self.conv3 = nn.Conv2d(width, width * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(width * self.expansion)

        self.relu = nn.ReLU(inplace=True)

        # Optional 1x1 conv + BN on the skip path when the input and output shapes differ
        self.downsample = downsample

    def forward(self, x):
        identity = x  # keep the original input for the skip connection

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))  # no ReLU yet, it comes after the addition

        # If the shape changed, reshape the identity so the two can be added
        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity  # the residual (skip) connection
        return self.relu(out)

In [ ]:
class ResNet50(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.in_channels = 64  # channel count going into the next block; updated as stages are built

        # Stem: 7x7 conv with stride 2 takes the 224x224 image down to 112x112
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        # Max-pool halves it again, 112x112 -> 56x56
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # The 4 stages of Bottleneck blocks: [3, 4, 6, 3] = 16 blocks
        self.layer1 = self._make_layer(64,  blocks=3, stride=1)  # 56x56,  256 channels out
        self.layer2 = self._make_layer(128, blocks=4, stride=2)  # 28x28,  512 channels out
        self.layer3 = self._make_layer(256, blocks=6, stride=2)  # 14x14, 1024 channels out
        self.layer4 = self._make_layer(512, blocks=3, stride=2)  # 7x7,   2048 channels out

        # Average each of the 2048 feature maps down to one number
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # Final classifier: 2048 features -> 2 classes (other activity / phone use)
        self.fc = nn.Linear(512 * Bottleneck.expansion, num_classes)

        self._init_weights()

    def _make_layer(self, width, blocks, stride):
        downsample = None
        out_channels = width * Bottleneck.expansion

        # The first block of a stage changes the shape, so the skip path needs reshaping too
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        # First block: does the downsampling
        layers = [Bottleneck(self.in_channels, width, stride, downsample)]
        self.in_channels = out_channels

        # Remaining blocks: same shape in and out, plain identity skip
        for _ in range(1, blocks):
            layers.append(Bottleneck(self.in_channels, width))

        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # Kaiming (He) init, designed for layers followed by ReLU
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                # BN starts as "do nothing": scale 1, shift 0
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))  # stem
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)         # [batch, 2048, 1, 1]
        x = torch.flatten(x, 1)     # [batch, 2048]
        return self.fc(x)           # [batch, 2] raw scores (logits)

In [ ]:
# Build the model and move it to the GPU/CPU picked in cell 7
model = ResNet50(num_classes=2).to(device)

# Take one batch from Henry's training loader and move it to the same device
images, labels = next(iter(train_loader))
images = images.to(device)

# Pass the batch through stage by stage, printing the shape after each part
model.eval()           # eval mode so BatchNorm doesn't update its running stats during this check
with torch.no_grad():  # no gradients needed, which saves memory
    x = images
    print("Input:  ", tuple(x.shape))
    x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
    print("Stem:   ", tuple(x.shape))
    for name in ["layer1", "layer2", "layer3", "layer4"]:
        x = getattr(model, name)(x)
        print(f"{name}: ", tuple(x.shape))

    # Then run the full forward pass to check the final output
    output = model(images)
print("Output: ", tuple(output.shape))

# The output must be one row per image and 2 columns (one score per class)
assert output.shape == (images.shape[0], 2), "Output shape is wrong"

# Count parameters: total, and how many will be updated during training
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")